# Module 1 — SOLUTION: Chunking & Embeddings

> **Instructor note:** This is the complete solution. Share after the exercise session.

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    SentenceTransformersTokenTextSplitter,
)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

with open("../data/raw/corpus.json") as f:
    corpus = json.load(f)

sample_doc = corpus[0]
sample_text = sample_doc["content"]
EMBED_MODEL = "paraphrase-multilingual-mpnet-base-v2"

print(f"Loaded {len(corpus)} documents. Sample: '{sample_doc['title']}'")

In [ ]:
# ── Strategy A: RecursiveCharacterTextSplitter ──────────────────────────────
splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks_recursive = splitter_recursive.split_text(sample_text)

# ── Strategy B: CharacterTextSplitter ───────────────────────────────────────
splitter_fixed = CharacterTextSplitter(
    separator=" ",
    chunk_size=800,
    chunk_overlap=100,
)
chunks_fixed = splitter_fixed.split_text(sample_text)

# ── Strategy C: SentenceTransformersTokenTextSplitter ───────────────────────
splitter_tokens = SentenceTransformersTokenTextSplitter(
    model_name=EMBED_MODEL,
    chunk_size=100,
    chunk_overlap=10,
)
chunks_tokens = splitter_tokens.split_text(sample_text)

for name, chunks in [("Recursive", chunks_recursive), ("Fixed", chunks_fixed), ("Tokens", chunks_tokens)]:
    sizes = [len(c) for c in chunks]
    print(f"{name}: {len(chunks)} chunks, avg={sum(sizes)/len(sizes):.0f} chars, min={min(sizes)}, max={max(sizes)}")

In [ ]:
# Visualise chunk size distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
strategies = [
    ("A — Recursive\n(800 chars)", chunks_recursive),
    ("B — Fixed-size\n(800 chars)", chunks_fixed),
    ("C — Token-aware\n(100 tokens)", chunks_tokens),
]
for ax, (title, chunks) in zip(axes, strategies):
    counts = [len(c) for c in chunks]
    ax.hist(counts, bins=20, color="steelblue", edgecolor="white")
    ax.set_title(f"{title}\nn={len(chunks)}, avg={sum(counts)/len(counts):.0f} chars")
    ax.set_xlabel("Characters per chunk")
plt.tight_layout()
plt.show()

In [ ]:
# Load multilingual model
model = SentenceTransformer(EMBED_MODEL)
print(f"Model loaded. Dim: {model.get_sentence_embedding_dimension()}, max tokens: {model.max_seq_length}")

In [ ]:
# Multilingual similarity heatmap
sentences = [
    "L'assureur prend en charge les frais d'hospitalisation.",
    "De verzekeraar dekt de ziekenhuiskosten.",
    "The insurer covers hospitalisation costs.",
    "Le remboursement est limité à 30 jours par année civile.",
    "Zijn de premies fiscaal aftrekbaar?",
]
embeddings = model.encode(sentences)
sim_matrix = cosine_similarity(embeddings)
labels = [s[:45] + "..." for s in sentences]

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(sim_matrix, annot=True, fmt=".2f",
            xticklabels=labels, yticklabels=labels,
            cmap="coolwarm", vmin=0, vmax=1, ax=ax)
plt.title("Cosine Similarity — Multilingual (FR / NL / EN)")
plt.tight_layout()
plt.show()

In [ ]:
# Exercise 1.4 solution — Batch embed all chunks with RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

all_chunks = []
for doc in corpus:
    doc_chunks = splitter.split_text(doc["content"])
    for i, text in enumerate(doc_chunks):
        all_chunks.append({
            "text": text,
            "source": doc["title"],
            "id": f"{doc['title']}_{i}",
        })

texts = [c["text"] for c in all_chunks]
print(f"Total chunks: {len(texts)}")

start = time.time()
all_embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)
elapsed = time.time() - start

print(f"\nEmbedded {len(texts)} chunks in {elapsed:.1f}s")
print(f"Embedding matrix shape: {all_embeddings.shape}")
print(f"Throughput: {len(texts)/elapsed:.0f} chunks/sec")